# 19 — ImprovedCNN1D: 5-seed Ensemble

nb18 (3-seed, LB=17.73) の拡張。N_SEEDS=5 で分散をさらに低減。

Seeds: [42, 123, 456, 789, 999]

Reference nb18 3-seed: [13.12, 20.65, 46.71, 18.98, 10.35] mean=21.96%  LB=17.73

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T  = 200.0
N_SEEDS = 5
SEEDS   = [42, 123, 456, 789, 999]

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'N_SEEDS={N_SEEDS}  Seeds={SEEDS}')
print(f'Total CV fits: {N_SEEDS * len(SPLITS)}  Test fits: {N_SEEDS}')

In [ ]:
class ImprovedCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.conv3     = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool1d(16)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 16, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.block12(x.unsqueeze(1))
        h = torch.relu(self.conv3(h) + self.shortcut3(h))
        h = self.pool(h)
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

n_params = sum(p.numel() for p in ImprovedCNN1D().parameters())
print(f'ImprovedCNN1D: {n_params:,} params')

def preprocess(R):
    A = R.copy().astype(float)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)

X_pp    = preprocess(X_raw)
X_pp_te = preprocess(X_test_raw)

def train_one(Xtr, ytr, Xva, yva, seed, n_epochs=100, batch=32, lr=1e-3, patience=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr).astype(np.float32)
    Xva_s = sc.transform(Xva).astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model = ImprovedCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.HuberLoss(delta=10.0)
    best_val, best_state, best_preds = float('inf'), None, None
    no_improve = 0
    stop_ep = n_epochs
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(Xva_t)
            val_loss = crit(val_pred, yva_t).item()
        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_preds = val_pred.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            stop_ep = epoch + 1
            break
    model.load_state_dict(best_state)
    return model, sc, best_preds, stop_ep

def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan
def rmse_all(yt, yp): return float(np.sqrt(np.mean((yt-yp)**2)))

print('Ready.')

## Multi-seed GroupKFold CV (N_SEEDS=5)

In [ ]:
print(f'=== Multi-seed CV (N_SEEDS={N_SEEDS}) ===')
print('Fold | Seed | stop_ep | RMSE_le170')
print('-' * 45)

fold_results  = []
oof_y_all     = []
oof_p_avg_all = []

for fi, (tr, va) in enumerate(SPLITS):
    Xtr, Xva = X_pp[tr], X_pp[va]
    ytr, yva = y[tr],    y[va]
    seed_preds = []
    stop_eps   = []
    for seed in SEEDS:
        _, _, pred_s, stop_ep = train_one(Xtr, ytr, Xva, yva, seed=seed)
        r_s = rmse_le(yva, pred_s)
        print(f'  {fi+1}  | {seed:3d}  |   {stop_ep:3d}   | {r_s:.2f}%')
        seed_preds.append(pred_s)
        stop_eps.append(stop_ep)
    avg_pred  = np.mean(seed_preds, axis=0)
    r_avg_le  = rmse_le(yva, avg_pred)
    r_avg_all = rmse_all(yva, avg_pred)
    fold_results.append({
        'fold': fi+1,
        'RMSE_le170': round(r_avg_le, 2),
        'RMSE_all':   round(r_avg_all, 2),
        'stop_eps':   stop_eps,
        'avg_stop':   int(np.mean(stop_eps)),
    })
    oof_y_all.append(yva)
    oof_p_avg_all.append(avg_pred)
    print(f'  Fold {fi+1} AVG -> RMSE_le170={r_avg_le:.2f}%  RMSE_all={r_avg_all:.2f}%')
    print('-' * 45)

oof_y = np.concatenate(oof_y_all)
oof_p = np.concatenate(oof_p_avg_all)
mean_le  = np.mean([r['RMSE_le170'] for r in fold_results])
mean_all = np.mean([r['RMSE_all']   for r in fold_results])

print()
print(f'CV mean RMSE_le170 : {mean_le:.2f}%')
print(f'CV mean RMSE_all   : {mean_all:.2f}%')
print(f'Folds RMSE_le170   : {[r["RMSE_le170"] for r in fold_results]}')
print()
print('OOF distribution:')
print(f'  min={oof_p.min():.1f}  mean={oof_p.mean():.1f}  max={oof_p.max():.1f}  std={oof_p.std():.1f}')
print(f'  neg: {(oof_p<0).sum()}  >170: {(oof_p>170).sum()}')
print()
print('vs nb18 3-seed:')
ref18 = [13.12, 20.65, 46.71, 18.98, 10.35]
for fi, (r, prev) in enumerate(zip([r['RMSE_le170'] for r in fold_results], ref18)):
    diff = r - prev
    mark = '+' if diff < -0.5 else ('-' if diff > 0.5 else '=')
    print(f'  Fold {fi+1}: {r:.2f}%  (3-seed={prev:.2f}%, diff={diff:+.2f}) [{mark}]')
print(f'  Mean : {mean_le:.2f}%  (3-seed=21.96%, diff={mean_le-21.96:+.2f})')

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ref18 = [13.12, 20.65, 46.71, 18.98, 10.35]
ens19 = [r['RMSE_le170'] for r in fold_results]
x = np.arange(5)
ax.bar(x - 0.2, ref18, 0.4, label='3-seed (nb18)', alpha=0.7)
ax.bar(x + 0.2, ens19, 0.4, label='5-seed (nb19)', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_ylabel('RMSE_le170 (%)')
ax.set_title('3-seed vs 5-seed ensemble')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
mask = oof_y <= 170
ax.scatter(oof_y[mask], oof_p[mask], s=6, alpha=0.4)
lim = max(oof_y[mask].max(), oof_p[mask].max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
ax.set_xlabel('Actual (%)'); ax.set_ylabel('Predicted (%)')
ax.set_title(f'OOF scatter (RMSE_le170={mean_le:.2f}%)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/nb19_ensemble5_cv.png', dpi=110)
plt.close()
print('Saved: results/nb19_ensemble5_cv.png')

## Full Train → Test Prediction (5-seed ensemble)

In [ ]:
print('=== Test predictions (5-seed ensemble on full train) ===')

avg_stop_global = int(np.mean([r['avg_stop'] for r in fold_results]))
print(f'Target epochs: {avg_stop_global}')

sc_full = StandardScaler()
Xtr_s   = sc_full.fit_transform(X_pp).astype(np.float32)
Xte_s   = sc_full.transform(X_pp_te).astype(np.float32)
Xtr_t   = torch.from_numpy(Xtr_s).to(DEVICE)
ytr_t   = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

te_seed_preds = []
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_s  = ImprovedCNN1D().to(DEVICE)
    opt_s    = torch.optim.Adam(model_s.parameters(), lr=1e-3)
    sched_s  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=100)
    crit_s   = nn.HuberLoss(delta=10.0)
    loader_s = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)
    for ep in range(avg_stop_global):
        model_s.train()
        for xb, yb in loader_s:
            loss = crit_s(model_s(xb), yb)
            opt_s.zero_grad(); loss.backward(); opt_s.step()
        sched_s.step()
    model_s.eval()
    with torch.no_grad():
        Xte_t = torch.from_numpy(Xte_s).to(DEVICE)
        pred_s = model_s(Xte_t).cpu().numpy()
    te_seed_preds.append(pred_s)
    tr_rmse = torch.sqrt(torch.mean((model_s(Xtr_t) - ytr_t)**2)).item()
    print(f'  seed={seed}: train_rmse={tr_rmse:.2f}')

te_pred = np.clip(np.mean(te_seed_preds, axis=0), 0, CLIP_T)
print(f'Test (5-seed): min={te_pred.min():.1f}  mean={te_pred.mean():.1f}  '
      f'max={te_pred.max():.1f}  >170: {(te_pred>170).sum()}')

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

make_submission(test_meta, te_pred, '../submissions/sub_cnn_ensemble5.csv')
print('Saved: submissions/sub_cnn_ensemble5.csv')

print()
print('=== Final Summary ===')
print(f'CV folds : {[r["RMSE_le170"] for r in fold_results]}')
print(f'CV mean  : {mean_le:.2f}%')
print(f'Test mean: {te_pred.mean():.1f}%')
print()
print('All CNN variants (CV / test_mean):')
print(f'  SmallCNN+MSE        : 25.78% / 54.8')
print(f'  SmallCNN+Huber(1)   : 20.92% / 60.9  LB=?')
print(f'  ImprovedCNN single  : 23.69% / 45.7  LB=?')
print(f'  ImprovedCNN 3-seed  : 21.96% / 44.8  LB=17.73  <-- confirmed')
print(f'  ImprovedCNN 5-seed  : {mean_le:.2f}% / {te_pred.mean():.1f}  LB=?')
print()
print('LB calibration (CNN): CV=21.96 -> LB=17.73')